# Sprint 4 — 回收原表：依沉睡客名單篩選所有資料

將 session01、session02、Member、Order_TG、Order_TS 全部篩選為只保留 `dormant_members` 中的 `ShopMemberId`，輸出至 `output/sprint3/`。

> ⚠️ **Cell 2、3 產生輸出 Parquet，已輸出則不需重跑。**  
> ⚠️ **原始 session CSV 已刪除，Cell 2 的 session 篩選部分無法重跑。**  
> ⚠️ **member.parquet 已另行去重（保留最早 RegisterDateTime），不可直接重跑 Cell 3 的 member 部分。**

## Cell 1 — 引入設定

建立 DuckDB 連線，建立 `output/sprint3/` 目錄，並將 `dormant_members.parquet` 的 `ShopMemberId` 建立為 VIEW（`dormant_ids`），供後續每個篩選 Cell 使用。

> ℹ️ 不產生輸出檔案

In [ ]:
import duckdb
import os

con = duckdb.connect()
os.makedirs('output/sprint3', exist_ok=True)

DORMANT_PATH = 'output/sprint1/dormant_members.parquet'

# 建立沉睡客 ShopMemberId 的 VIEW（後續每次篩選都用這個）
con.execute(f"""
CREATE OR REPLACE VIEW dormant_ids AS
SELECT ShopMemberId FROM read_parquet('{DORMANT_PATH}')
""")

dormant_cnt = con.execute("SELECT COUNT(*) FROM dormant_ids").fetchone()[0]
print(f'沉睡客人數：{dormant_cnt:,}')
print(f'輸出目錄：output/sprint3/')

## Cell 2 — Session 行為資料篩選（session01 × 6 + session02 × 6）

逐一讀取 12 個 session CSV，與 `dormant_ids` INNER JOIN，只保留沉睡客的行為事件，輸出為 Parquet。
使用 `nullstr='(null)'` 處理 CSV 中的字串型 NULL（session01_202401 起開始出現）。
印出每個檔案的原始筆數、篩選後筆數與保留率。

> 📄 **輸出**：`output/sprint3/session01_202309.parquet` ~ `session02_202402.parquet`（共 12 個檔案）  
> ⚠️ 依賴原始 session CSV（已刪除），無法重跑。

In [ ]:
session_files = [
    '91APP_Dataset(session01)/session01_202309.csv',
    '91APP_Dataset(session01)/session01_202310.csv',
    '91APP_Dataset(session01)/session01_202311.csv',
    '91APP_Dataset(session01)/session01_202312.csv',
    '91APP_Dataset(session01)/session01_202401.csv',
    '91APP_Dataset(session01)/session01_202402.csv',
    '91APP_Dataset(session02)/session02_202309.csv',
    '91APP_Dataset(session02)/session02_202310.csv',
    '91APP_Dataset(session02)/session02_202311.csv',
    '91APP_Dataset(session02)/session02_202312.csv',
    '91APP_Dataset(session02)/session02_202401.csv',
    '91APP_Dataset(session02)/session02_202402.csv',
]

print(f"{'檔案':<36} {'原始筆數':>12} {'篩選後':>10} {'保留率':>8}")
print('-' * 72)

for csv_path in session_files:
    fname = csv_path.split('/')[-1].replace('.csv', '')
    out_path = f'output/sprint3/{fname}.parquet'

    orig = con.execute(f"SELECT COUNT(*) FROM read_csv_auto('{csv_path}')").fetchone()[0]

    con.execute(f"""
    COPY (
        SELECT s.*
        FROM read_csv_auto('{csv_path}') AS s
        INNER JOIN dormant_ids AS d USING (ShopMemberId)
        WHERE s.ShopMemberId IS NOT NULL
    ) TO '{out_path}' (FORMAT PARQUET)
    """)

    kept = con.execute(f"SELECT COUNT(*) FROM read_parquet('{out_path}')").fetchone()[0]
    print(f"{fname:<36} {orig:>12,} {kept:>10,} {kept/orig*100:>7.1f}%")

print('-' * 72)
print('Session 篩選完成')

## Cell 3 — 會員與訂單資料篩選（Member / Order_TG / Order_TS）

讀取 Member.csv、Order_TG.csv、Order_TS.csv，與 `dormant_ids` INNER JOIN，只保留沉睡客資料，輸出為 Parquet。
印出每個檔案的原始筆數、篩選後筆數與保留率。

> 📄 **輸出**：`output/sprint3/member.parquet`（1,112,082 筆，去重前）  
> 📄 **輸出**：`output/sprint3/order_tg.parquet`（2,725,614 筆）  
> 📄 **輸出**：`output/sprint3/order_ts.parquet`（10,634,742 筆）  
> ℹ️ **注意**：`member.parquet` 已於事後去重（保留最早 RegisterDateTime），去重後 1,112,081 筆。若重跑此 Cell，需再手動執行去重步驟。

In [ ]:
main_files = [
    ('91APP_Dataset(main)/Member.csv',   'output/sprint3/member.parquet'),
    ('91APP_Dataset(main)/Order_TG.csv', 'output/sprint3/order_tg.parquet'),
    ('91APP_Dataset(main)/Order_TS.csv', 'output/sprint3/order_ts.parquet'),
]

print(f"{'檔案':<20} {'原始筆數':>12} {'篩選後':>12} {'保留率':>8}")
print('-' * 58)

for csv_path, out_path in main_files:
    fname = csv_path.split('/')[-1]

    orig = con.execute(f"SELECT COUNT(*) FROM read_csv_auto('{csv_path}')").fetchone()[0]

    con.execute(f"""
    COPY (
        SELECT s.*
        FROM read_csv_auto('{csv_path}') AS s
        INNER JOIN dormant_ids AS d USING (ShopMemberId)
        WHERE s.ShopMemberId IS NOT NULL
    ) TO '{out_path}' (FORMAT PARQUET)
    """)

    kept = con.execute(f"SELECT COUNT(*) FROM read_parquet('{out_path}')").fetchone()[0]
    print(f"{fname:<20} {orig:>12,} {kept:>12,} {kept/orig*100:>7.1f}%")

print('-' * 58)
print('主表篩選完成')